<a href="https://colab.research.google.com/github/Shavkatov-dev/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Menu Detector!")

Menu Detector!


In [2]:

#--------------
# Imoport Libararies
#--------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import Dataset
from PIL import UnidentifiedImageError
import os
import numpy as np

In [15]:
# Define dataset path
DATASET_PATH = '/content/drive/MyDrive/kaggle/images/'
print("Dataset Path: ", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    "chocolate_cake" : "dessert",  # label grouping | class consolidation
    "cheesecake" : "dessert",  # label grouping | class consolidation
    "bibimbap": "bibimbap",
    "baklava" : "dessert", # label grouping | class consolidation
    'lasagna': 'salad',
    'pilaf_images': 'pilaf_images'
}


CLASSES = ['hamburger', 'hot_dog', 'dessert','salad', 'pilaf_images', "bibimbap"]
COLORS = ['white','black', 'red', 'blue']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
COLORS_TO_IDX = {cls: i for i, cls in enumerate(COLORS)}
print(COLORS_TO_IDX)
NUM_CLASS = len(CLASSES)
print(CLASS_TO_IDX)
print("Number of Classes: ", NUM_CLASS)


transform = transforms.Compose([
    transforms.Resize((224,224)), #image resizing
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#300x400, 512x512
# H, W, C => H, W, C
# Normalize
# pixel = (pixel - mean) / std

Dataset Path:  /content/drive/MyDrive/kaggle/images/
{'white': 0, 'black': 1, 'red': 2, 'blue': 3}
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'salad': 3, 'pilaf_images': 4, 'bibimbap': 5}
Number of Classes:  6


In [16]:
# ----------
# Custom Dataset class
# ----------

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.transform = transform
    self.labels = labels

  def __len__(self):
    print("images_length:", len(self.images))
    return len(self.images)

  def __getitem__ (self, idx):
    img_path = self.images[idx]
    print('images_path', img_path)
    label = self.labels[idx]
    print('label', label)
    try:
      image = Image.open(img_path).convert('RGB') # 3 channel
    except (UnidentifiedImageError, OSError):
      print(f'Skipping broken image: {img_path}')
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
      return image, label


In [19]:
# -----------------------------
# Gather and Split Data
# -----------------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class) # /content/drive/MyDrive/kaggle/images/apple-pie
    print('class_path:', class_path)
    if not os.path.exists(class_path): # Corrected line: removed extra '.path'
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img) # /content/drive/MyDrive/kaggle/images/apple-pie/1001.jpg
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split] #1000 => 800 | split | 200
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels, transform=transform) # Corrected: Added transform=transform
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/kaggle/images/hamburger
class_path: /content/drive/MyDrive/kaggle/images/hot_dog
class_path: /content/drive/MyDrive/kaggle/images/chocolate_cake
class_path: /content/drive/MyDrive/kaggle/images/cheesecake
class_path: /content/drive/MyDrive/kaggle/images/bibimbap
class_path: /content/drive/MyDrive/kaggle/images/baklava
class_path: /content/drive/MyDrive/kaggle/images/lasagna
class_path: /content/drive/MyDrive/kaggle/images/pilaf_images
images_length: 5674
5674
images_path /content/drive/MyDrive/kaggle/images/cheesecake/165910.jpg
label 2
